# BIAS-PROFS データセットのグラフ表示画像およびラベルCSVの作成
以下の指標を使用します（相関係数 $r \approx 0$）
- $r$：`NAKH900107`
- $\theta$：`PALJ810108`

## 1. 必要ライブラリのインポート

In [1]:
from Bio import SeqIO
from PIL import Image
from tqdm import tqdm
import csv
import numpy as np
import os
import shutil

## 2. BIAS-PROFS データセットの読み込み

In [5]:
biasprofs_path = "../data/gds_dataset"  # BIAS-PROFS データセットの保管場所
paths = [os.path.join(biasprofs_path, f) for f in os.listdir(biasprofs_path) if f.endswith(".txt")]  # パスのリスト

In [1]:
aa_list = list("ACDEFGHIKLMNPQRSTVWY")
exclude_aa = list("BJOUXZ")

def check_valid_seq(seq):
    """
    有効なタンパク質かどうかを返す関数
    :param seq: タンパク質配列
    :return: 有効なタンパク質ならば True そうでないならば False
    """
    if len(seq) >= 1000:  # 配列長が 1000 以上のものを除外
        return False

    for aa in exclude_aa:  # BJOUXZ が含まれていたら除外
        if aa in seq:
            return False

    return True

## 3. グラフ表示画像の作成

In [7]:
SIZE = 224  # 画像サイズ
PADDING = 3  # パディング
LINEWIDTH = 2  # 線の太さ
IMAGE_SAVE_DIR = "../images/bias-profs-cor0"  # 画像の保存場所

os.makedirs(IMAGE_SAVE_DIR, exist_ok=True)  # ディレクトリの作成

# アミノ酸のベクトル（詳細は論文参照）
amino_vec = {
    "A": {'x': -3.021716477288284, 'y': 4.463331662657894},
    "C": {'x': -0.36233629033978615, 'y': 0.7799438522757918},
    "D": {'x': 0.14049283824157704, 'y': 3.066783618451557},
    "E": {'x': -2.6589809331329617, 'y': 0.4688500797007118},
    "F": {'x': 0.8679844072348316, 'y': 6.2803027848024335},
    "G": {'x': 6.420946549639596, 'y': 1.1321861183883857},
    "H": {'x': -0.15301086574475964, 'y': 2.224744406659794},
    "I": {'x': 5.189997855903639, 'y': 8.477471454137468},
    "K": {'x': -7.509932773989893e-16, 'y': 4.67},
    "L": {'x': -8.235118420374606, 'y': 9.589182686882488},
    "M": {'x': 1.47367037279666, 'y': 3.372046208512178},
    "N": {'x': 3.9584804546017334, 'y': 6.14544811145095},
    "P": {'x': 3.549682926508577, 'y': 0.7100359999700732},
    "Q": {'x': -0.2637976987810208, 'y': 2.2948879654828116},
    "R": {'x': 1.0660486448932056, 'y': 2.599930823449224},
    "S": {'x': 0.49677070313544736, 'y': 7.222936997406686},
    "T": {'x': 3.8190475950097964, 'y': 3.8740773697810806},
    "V": {'x': -1.402255015870612, 'y': 6.018810586026587},
    "W": {'x': 1.386304622594539, 'y': 0.8762188615711333},
    "Y": {'x': 3.892352452209962, 'y': 3.7717359912611985}
}

def map_range(x, a, b, c, d):
    """
    変数 $x$ を 範囲 [$a$, $b$] から [$c$, $d$] にリマップします
    :param x: 変換する対象
    :param a: 変換前の下端
    :param b: 変換前の上端
    :param c: 変換後の下端
    :param d: 変換後の上端
    :return: リマップした値
    """
    return (x - a) / (b - a) * (d - c) + c


def draw_point(canvas, x: int, y: int, width: int, intensity: int):
    """
    点を描画します
    :param canvas: numpy の2次元配列
    :param x: 座標 $x$
    :param y: 座標 $y$
    :param width: 点の幅
    :param intensity: 点の白色の強度
    :return:
    """

    size = canvas.shape[0]

    for i in range(-width, width + 1):
        yi = y + i

        if 0 <= yi < size:
            for j in range(-width, width + 1):
                xi = x + j

                if 0 <= xi < size:
                    r = np.sqrt(i ** 2 + j ** 2)  # 中心からの距離

                    if r <= width:  # 半径内部かどうかを判定
                        val = int(intensity * (1 - r / width))  # 直線的に色を変化
                        # val = int((intensity / (r_0 + 1)) - 1)  # 中心から反比例するように色を変化

                        if val > canvas[yi, xi]:  # 画素よりも大きい値であれば上書き
                            canvas[yi, xi] = val


def drawline_with_bresenham_algorithm(canvas, x0: int, y0: int, x1: int, y1: int, width: int, intensity: int):
    """
    ブレゼンハムのアルゴリズムで2点間の直線を描画します
    :param canvas: numpy の2次元配列
    :param x0: 点 $x_0$
    :param y0: 点 $y_0$
    :param x1: 点 $x_1$
    :param y1: 点 $y_1$
    :param width 直線の幅
    :return: None
    """

    # ここで float を int に変換
    x0 = round(x0)
    y0 = round(y0)
    x1 = round(x1)
    y1 = round(y1)

    dx = abs(x1 - x0)
    dy = abs(y1 - y0)
    sx = 1 if x0 < x1 else -1
    sy = 1 if y0 < y1 else -1
    err = dx - dy

    while True:
        draw_point(canvas, x0, y0, width, intensity)

        if x0 == x1 and y0 == y1:
            break
        e2 = 2 * err

        if e2 > -dy:
            err -= dy
            x0 += sx

        if e2 < dx:
            err += dx
            y0 += sy

    draw_point(canvas, x1, y1, width, intensity)


def generate_graph(seq, accession_number, size, padding, width):
    """
    アミノ酸配列からグラフ表示画像を作成します
    :param seq: アミノ酸配列
    :param accession_number: アクセッション番号
    :param size: 画像サイズ
    :param padding: 画像のパディング
    :param width: 線幅
    :return:
    """

    x = y = 0
    x_min = x_max = 0
    y_min = y_max = 0

    points = [{"x": 0, "y": 0}]

    for c in seq:
        x += amino_vec[c]["x"]
        y += amino_vec[c]["y"]  # 人間が見るXY平面にする

        points.append({"x": x, "y": y})

        x_min, x_max = min(x, x_min), max(x, x_max)
        y_min, y_max = min(y, y_min), max(y, y_max)

    mapped_points = [
        {
            "x": map_range(point["x"], x_min, x_max, padding, (size - padding)),
            "y": size - map_range(point["y"], y_min, y_max, padding, (size - padding))  # xy 平面にするために反転
        }
        for point in points
    ]

    canvas = np.zeros((size, size), dtype=np.uint8)
    n_segments = len(mapped_points) - 1

    for i in range(n_segments):
        start, end = mapped_points[i], mapped_points[i + 1]

        progress = i / n_segments
        intensity = int(255 * progress)
        # draw.line([(start["x"], start["y"]), (end["x"], end["y"])], fill=255, width=3)
        drawline_with_bresenham_algorithm(canvas, start["x"], start["y"], end["x"], end["y"], width, intensity)

    img = Image.fromarray(canvas, mode="L")  # numpy 配列から画像を生成
    img.save(os.path.join(IMAGE_SAVE_DIR, f"{accession_number}.png"))

In [8]:
n = 1

for path in tqdm(paths):
    cls_name = os.path.basename(path).split("Class")[1][0]  # クラス名の取得

    with open(path, "r", encoding="utf-8") as handle:
        for record in SeqIO.parse(handle, "fasta"):
            if check_valid_seq(record.seq):  # 有効な配列かどうか
                seq = record.seq
                generate_graph(seq, n, SIZE, PADDING, LINEWIDTH)  # グラフ表示画像の作成

                n += 1

100%|██████████| 87/87 [06:29<00:00,  4.48s/it]
